# Лабораторна робота 1

Ручний backpropagation для мережі `4 -> 8 -> 3` на Iris.

Усі обчислення NumPy виконуються у `float64`. Спочатку дані діляться по класах у пропорції 35/15, потім стандартизуються за параметрами train.

У notebook запускається та сама реалізація, що й у `backprop.py`. Основні кроки: `z1 = X @ W1 + b1`, `a1 = ReLU(z1)`, `z2 = a1 @ W2 + b2`, далі обчислюється середня cross-entropy.

Зворотний прохід починається з `dz2 = (softmax(z2) - one_hot(y)) / N`; після цього обчислюються градієнти `W2`, `b2`, `W1` і `b1`.

In [1]:
from backprop import (
    backward,
    forward,
    initial_parameters,
    run_checks,
    split_iris,
)

x_train, y_train, x_test, y_test = split_iris()
parameters = initial_parameters()
loss, cache = forward(parameters, x_train, y_train)
gradients = backward(parameters, cache)

print('train:', x_train.shape, 'test:', x_test.shape)
print('class counts:', [int((y_train == i).sum()) for i in range(3)])
print('loss:', loss)
print('shapes:', parameters.W1.shape, parameters.b1.shape, parameters.W2.shape, parameters.b2.shape)

train: (105, 4) test: (45, 4)
class counts: [35, 35, 35]
loss: 1.4562007801142816
shapes: (4, 8) (8,) (8, 3) (3,)


## Звірка з PyTorch

In [2]:
result = run_checks(False)
print('NumPy loss:', result['numpy_loss'])
print('PyTorch loss:', result['torch_loss'])
print('loss difference:', result['loss_absolute_difference'])
print('gradient differences:')
for name, value in result['gradient_max_absolute_differences'].items():
    print(f'  {name}: {value:.3e}')
print('all compared values finite:', result['finite_values_passed'])
print('PyTorch check:', result['torch_check_passed'])

NumPy loss: 1.4562007801142816
PyTorch loss: 1.4562007801142818
loss difference: 2.220446049250313e-16
gradient differences:
  W1: 2.776e-17
  b1: 3.990e-17
  W2: 5.551e-17
  b2: 1.388e-16
all compared values finite: True
PyTorch check: True


## Чисельна похідна

In [3]:
for row in result['numerical_checks']:
    print(
        row['parameter'],
        'manual =', f"{row['manual']:.12f}",
        'numerical =', f"{row['numerical']:.12f}",
        'difference =', f"{row['absolute_difference']:.3e}",
        'passed =', row['passed'],
    )
print('numerical check:', result['numerical_check_passed'])

W1(0, 0) manual = -0.019920184331 numerical = -0.019920184391 difference = 6.034e-11 passed = True
b1(0,) manual = -0.070566111584 numerical = -0.070566111621 difference = 3.633e-11 passed = True
W2(0, 0) manual = 0.135230208605 numerical = 0.135230208698 difference = 9.218e-11 passed = True
b2(0,) manual = 0.085521644842 numerical = 0.085521644988 difference = 1.460e-10 passed = True
numerical check: True


## Перевірка навмисної помилки

Якщо прибрати ділення `dz2` на кількість об'єктів, loss не змінюється, але градієнти стають неправильними.

In [4]:
wrong = run_checks(True)
print('loss:', wrong['numpy_loss'])
print('PyTorch check:', wrong['torch_check_passed'])
print('numerical check:', wrong['numerical_check_passed'])
print('error detected:', wrong['wrong_gradient_detected'])

loss: 1.4562007801142816
PyTorch check: False
numerical check: False
error detected: True


Висновок: ручні градієнти збігаються з PyTorch, а чисельна перевірка підтверджує похідні для чотирьох вибраних параметрів.